In [1]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import asyncio

try:
    from ib_async import IB, Stock, MarketOrder, LimitOrder
except ImportError:
    from ib_insync import IB, Stock, MarketOrder, LimitOrder



In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from ib_async import IB, Stock, MarketOrder, LimitOrder
except ImportError:
    from ib_insync import IB, Stock, MarketOrder, LimitOrder

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Market Data" / "Cleaned"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ORDERS_PARQUET = DATA_DIR / "orders_only.parquet"
ORDERS_CSV = DATA_DIR / "orders_only.csv"

HOST = "127.0.0.1"
PORT = 4004
CLIENT_ID = 107

SUBMIT_ORDERS = False
ORDER_TYPE = "MKT"
LIMIT_BUFFER = 0.002

MAX_ORDER_COUNT = 25
MAX_SINGLE_ORDER_DOLLARS = 350_000
MAX_TOTAL_TRADE_DOLLARS = 1_050_000

SUBMIT_ORDERS = True
ORDER_TYPE = "MKT"
BUY_ONLY = True

In [3]:
if ORDERS_PARQUET.exists():
    orders = pd.read_parquet(ORDERS_PARQUET)
else:
    orders = pd.read_csv(ORDERS_CSV)

orders["date"] = pd.to_datetime(orders["date"])
orders = orders.copy()

orders["delta_shares"] = orders["delta_shares"].astype(int)
orders = orders[orders["delta_shares"] != 0].copy()

orders["side"] = np.where(orders["delta_shares"] > 0, "BUY", "SELL")
orders["order_qty"] = orders["delta_shares"].abs().astype(int)

if BUY_ONLY:
    orders = orders[orders["side"] == "BUY"].copy()

orders = orders.sort_values(["estimated_trade_dollars"], ascending=False).reset_index(drop=True)

display(orders)
print("Order count:", len(orders))
print("Total estimated trade dollars:", orders["estimated_trade_dollars"].sum())

,date,symbol,bucket,close,target_weight,target_dollars,target_shares,current_shares,delta_shares,side,estimated_trade_dollars,abs_delta_shares,order_qty
0,2026-03-06,SPY,core_us,672.38,0.300000,300025.905000,446,0,446,BUY,299881.48,446,446
1,2026-03-06,QQQ,core_us,599.75,0.300000,300025.905000,500,0,500,BUY,299875.00,500,500
2,2026-03-06,VPU,us_sector,201.60,0.050000,50004.317500,248,0,248,BUY,49996.80,248,248
3,2026-03-06,VFH,us_sector,123.43,0.050000,50004.317500,405,0,405,BUY,49989.15,405,405
4,2026-03-06,VHT,us_sector,282.15,0.050000,50004.317500,177,0,177,BUY,49940.55,177,177
5,2026-03-06,VIS,us_sector,326.26,0.050000,50004.317500,153,0,153,BUY,49917.78,153,153
6,2026-03-06,VWO,international,54.47,0.033333,33336.211667,612,0,612,BUY,33335.64,612,612
7,2026-03-06,INDA,international,49.99,0.033333,33336.211667,666,0,666,BUY,33293.34,666,666
8,2026-03-06,VEA,international,65.28,0.033333,33336.211667,510,0,510,BUY,33292.80,510,510
9,2026-03-06,SHY,defensive,82.73,0.012500,12501.079375,151,0,151,BUY,12492.23,151,151


Order count: 18
Total estimated trade dollars: 999044.33


In [4]:
assert len(orders) <= MAX_ORDER_COUNT, "Too many orders"
assert orders["estimated_trade_dollars"].max() <= MAX_SINGLE_ORDER_DOLLARS, "Single order too large"
assert orders["estimated_trade_dollars"].sum() <= MAX_TOTAL_TRADE_DOLLARS, "Total trade dollars too large"

In [5]:
if SUBMIT_ORDERS:
    confirm = input("Type YES to submit paper orders: ").strip()
    if confirm != "YES":
        raise ValueError("Submission cancelled by user")
else:
    print("SUBMIT_ORDERS is False. Preview mode only.")

In [6]:
ib = IB()
await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)
print("Connected:", ib.isConnected())

Connected: True


In [7]:
summary = await ib.accountSummaryAsync()
summary_rows = [{"tag": x.tag, "value": x.value, "currency": x.currency} for x in summary]
summary_df = pd.DataFrame(summary_rows)

display(summary_df.head(20))

netliq_row = summary_df[summary_df["tag"] == "NetLiquidation"]
if not netliq_row.empty:
    print("NetLiquidation:", netliq_row.iloc[0]["value"], netliq_row.iloc[0]["currency"])

,tag,value,currency
0,AccountType,INDIVIDUAL,
1,Cushion,1,
2,DayTradesRemaining,-1,
3,DayTradesRemainingT+1,-1,
4,DayTradesRemainingT+2,-1,
5,DayTradesRemainingT+3,-1,
6,DayTradesRemainingT+4,-1,
7,LookAheadNextChange,0,
8,AccruedCash,86.35,USD
9,AvailableFunds,1000000.00,USD


NetLiquidation: 1000086.35 USD


In [8]:
async def make_contract(symbol: str):
    contract = Stock(symbol, "SMART", "USD")
    qualified = await ib.qualifyContractsAsync(contract)
    if not qualified:
        raise ValueError(f"Could not qualify contract for {symbol}")
    return qualified[0]

def make_order(side: str, qty: int, ref_price: float):
    if ORDER_TYPE == "MKT":
        return MarketOrder(side, qty, tif="DAY")
    elif ORDER_TYPE == "LMT":
        if side == "BUY":
            limit_price = round(ref_price * (1 + LIMIT_BUFFER), 2)
        else:
            limit_price = round(ref_price * (1 - LIMIT_BUFFER), 2)
        return LimitOrder(side, qty, limit_price, tif="DAY")
    else:
        raise ValueError("ORDER_TYPE must be 'MKT' or 'LMT'")

In [9]:
preview_rows = []

for _, row in orders.iterrows():
    preview_rows.append({
        "symbol": row["symbol"],
        "side": row["side"],
        "qty": int(row["order_qty"]),
        "ref_price": float(row["close"]),
        "estimated_trade_dollars": float(row["estimated_trade_dollars"]),
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

,symbol,side,qty,ref_price,estimated_trade_dollars
0,SPY,BUY,446,672.38,299881.48
1,QQQ,BUY,500,599.75,299875.00
2,VPU,BUY,248,201.60,49996.80
3,VFH,BUY,405,123.43,49989.15
4,VHT,BUY,177,282.15,49940.55
5,VIS,BUY,153,326.26,49917.78
6,VWO,BUY,612,54.47,33335.64
7,INDA,BUY,666,49.99,33293.34
8,VEA,BUY,510,65.28,33292.80
9,SHY,BUY,151,82.73,12492.23


In [10]:
submitted_trades = []

if SUBMIT_ORDERS:
    for _, row in orders.iterrows():
        symbol = row["symbol"]
        side = row["side"]
        qty = int(row["order_qty"])
        ref_price = float(row["close"])

        contract = await make_contract(symbol)
        order = make_order(side, qty, ref_price)

        trade = ib.placeOrder(contract, order)
        await asyncio.sleep(1.5)
        submitted_trades.append({
            "symbol": symbol,
            "side": side,
            "qty": qty,
            "ref_price": ref_price,
            "status": trade.orderStatus.status,
            "filled": float(trade.orderStatus.filled),
            "remaining": float(trade.orderStatus.remaining),
            "avgFillPrice": float(trade.orderStatus.avgFillPrice or 0),
            "permId": getattr(trade.orderStatus, "permId", None),
        })

        print(symbol, side, qty, trade.orderStatus.status)
else:
    print("Preview only. No orders submitted.")

SPY BUY 446 Filled
QQQ BUY 500 Filled
VPU BUY 248 Filled
VFH BUY 405 Filled
VHT BUY 177 Filled
VIS BUY 153 Filled
VWO BUY 612 Filled
INDA BUY 666 Filled
VEA BUY 510 Filled
SHY BUY 151 Filled
TLT BUY 141 Filled
BIL BUY 136 Filled
IEI BUY 104 Filled
UNG BUY 784 Filled
SLV BUY 131 Filled
GLD BUY 21 Filled
VNQ BUY 106 Filled
USO BUY 91 Filled


In [11]:
submitted_df = pd.DataFrame(submitted_trades)
display(submitted_df)

,symbol,side,qty,ref_price,status,filled,remaining,avgFillPrice,permId
0,SPY,BUY,446,672.38,Filled,446.0,0.0,668.71,778113992
1,QQQ,BUY,500,599.75,Filled,500.0,0.0,598.35,778113993
2,VPU,BUY,248,201.60,Filled,248.0,0.0,200.77,778113994
3,VFH,BUY,405,123.43,Filled,405.0,0.0,120.76,778113995
4,VHT,BUY,177,282.15,Filled,177.0,0.0,282.24,778113996
5,VIS,BUY,153,326.26,Filled,153.0,0.0,323.01,778113997
6,VWO,BUY,612,54.47,Filled,612.0,0.0,54.31,778113998
7,INDA,BUY,666,49.99,Filled,666.0,0.0,49.45,778113999
8,VEA,BUY,510,65.28,Filled,510.0,0.0,64.71,778114000
9,SHY,BUY,151,82.73,Filled,151.0,0.0,82.72,778114001


In [12]:
log_rows = []

if SUBMIT_ORDERS:
    await asyncio.sleep(1.5)

    for trade in ib.trades():
        contract_symbol = getattr(trade.contract, "symbol", None)
        for entry in getattr(trade, "log", []):
            log_rows.append({
                "symbol": contract_symbol,
                "time": entry.time,
                "status": entry.status,
                "message": entry.message,
                "errorCode": entry.errorCode,
            })

trade_log_df = pd.DataFrame(log_rows)
display(trade_log_df.tail(50))

,symbol,time,status,message,errorCode
35,VIS,2026-03-09 15:50:24.501291+00:00,Submitted,Fill 153.0@323.01,0
36,VIS,2026-03-09 15:50:24.501597+00:00,Filled,,0
37,VWO,2026-03-09 15:50:25.953776+00:00,PendingSubmit,,0
38,VWO,2026-03-09 15:50:25.997976+00:00,Submitted,,0
39,VWO,2026-03-09 15:50:26.042589+00:00,Submitted,Fill 612.0@54.31,0
40,VWO,2026-03-09 15:50:26.043294+00:00,Filled,,0
41,INDA,2026-03-09 15:50:27.543843+00:00,PendingSubmit,,0
42,INDA,2026-03-09 15:50:27.586407+00:00,PreSubmitted,,0
43,INDA,2026-03-09 15:50:27.628262+00:00,PreSubmitted,Fill 666.0@49.45,0
44,INDA,2026-03-09 15:50:27.628490+00:00,Filled,,0


In [13]:
positions_rows = []

for p in ib.positions():
    if getattr(p.contract, "secType", None) == "STK":
        positions_rows.append({
            "symbol": p.contract.symbol,
            "position": float(p.position),
            "avgCost": float(p.avgCost),
            "exchange": getattr(p.contract, "exchange", None),
            "currency": getattr(p.contract, "currency", None),
        })

positions_df = pd.DataFrame(positions_rows)
display(positions_df.sort_values("symbol") if not positions_df.empty else positions_df)

,symbol,position,avgCost,exchange,currency
11,BIL,136.0,91.467353,ARCA,USD
15,GLD,21.0,468.457619,ARCA,USD
12,IEI,104.0,119.429615,NASDAQ,USD
7,INDA,666.0,49.455000,BATS,USD
1,QQQ,500.0,598.355000,NASDAQ,USD
9,SHY,151.0,82.726623,NASDAQ,USD
14,SLV,131.0,76.447634,ARCA,USD
0,SPY,446.0,668.715000,ARCA,USD
10,TLT,141.0,88.797092,NASDAQ,USD
13,UNG,784.0,12.665000,ARCA,USD


In [14]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if not submitted_df.empty:
    submitted_path = DATA_DIR / f"paper_orders_submitted_{timestamp}.csv"
    submitted_df.to_csv(submitted_path, index=False)
    print(submitted_path)

if not trade_log_df.empty:
    trade_log_path = DATA_DIR / f"paper_trade_log_{timestamp}.csv"
    trade_log_df.to_csv(trade_log_path, index=False)
    print(trade_log_path)

positions_path = DATA_DIR / "paper_positions_snapshot.csv"
positions_df.to_csv(positions_path, index=False)
print(positions_path)

account_snapshot_path = DATA_DIR / f"paper_account_summary_{timestamp}.csv"
summary_df.to_csv(account_snapshot_path, index=False)
print(account_snapshot_path)

/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_orders_submitted_20260309_105046.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_trade_log_20260309_105046.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_positions_snapshot.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/paper_account_summary_20260309_105046.csv


In [15]:
ib.disconnect()
print("Disconnected")

Disconnected
